# Variational Autoencoder (VAE) from Scratch
## Learn probabilistic modeling and generative AI

VAE learns a **probability distribution** over the latent space, enabling:
- Smooth interpolation between samples
- Generating new realistic data
- Disentangled feature learning

Key innovation: Reparameterization trick for backprop through stochastic nodes.

---

In [ ]:
# CELL 1: Setup
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
BATCH_SIZE = 128; EPOCHS = 30; LATENT_DIM = 20; LR = 1e-3

transform = transforms.Compose([transforms.ToTensor()])
train_data = torchvision.datasets.MNIST('data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
print(f'Device: {device} | Dataset: {len(train_data)}')

In [ ]:
# CELL 2: VAE Architecture
class VAE(nn.Module):
    """
    Variational Autoencoder with reparameterization trick.
    
    Instead of encoding to a single point, we encode to
    mu (mean) and log_var (log variance) of a Gaussian distribution,
    then sample from it using: z = mu + std * epsilon
    """
    def __init__(self, latent_dim=20):
        super().__init__()
        # Encoder
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64*7*7, latent_dim)      # Mean
        self.fc_logvar = nn.Linear(64*7*7, latent_dim)   # Log variance
        
        # Decoder
        self.dec_fc = nn.Linear(latent_dim, 64*7*7)
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.enc_conv(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + std * epsilon"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)  # Sample from N(0,1)
        return mu + eps * std
    
    def decode(self, z):
        h = self.dec_fc(z).view(-1, 64, 7, 7)
        return self.dec_conv(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    """VAE Loss = Reconstruction (BCE) + KL Divergence"""
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    # KL divergence: pushes latent distribution toward N(0,1)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld

model = VAE(LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
print(f"VAE params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# CELL 3: Train VAE
losses = []
for epoch in range(EPOCHS):
    model.train(); total_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        recon, mu, logvar = model(images)
        loss = vae_loss(recon, images, mu, logvar)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(train_data)
    losses.append(avg)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg:.2f}")

plt.plot(losses, 'r-', linewidth=2)
plt.title('VAE Training Loss (ELBO)'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.savefig(OUTPUT_DIR / 'vae_training.png', dpi=150); plt.show()

In [ ]:
# CELL 4: Generate New Images (Sampling from Latent Space)
model.eval()
with torch.no_grad():
    # Sample from standard normal
    z = torch.randn(64, LATENT_DIM).to(device)
    generated = model.decode(z).cpu()

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('VAE Generated Digits (Sampled from N(0,1))', fontsize=14)
plt.savefig(OUTPUT_DIR / 'vae_generated.png', dpi=150); plt.show()

In [ ]:
# CELL 5: Latent Space Interpolation
model.eval()
with torch.no_grad():
    # Get two real images
    imgs, labels = next(iter(train_loader))
    mu1, _ = model.encode(imgs[0:1].to(device))
    mu2, _ = model.encode(imgs[1:2].to(device))
    
    # Interpolate in latent space
    n_steps = 10
    alphas = np.linspace(0, 1, n_steps)
    interp = torch.stack([mu1 * (1-a) + mu2 * a for a in alphas]).squeeze(1)
    decoded = model.decode(interp).cpu()

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(decoded[i].squeeze(), cmap='gray'); ax.axis('off')
plt.suptitle(f'Latent Space Interpolation: {labels[0].item()} -> {labels[1].item()}')
plt.savefig(OUTPUT_DIR / 'vae_interpolation.png', dpi=150); plt.show()

torch.save(model.state_dict(), OUTPUT_DIR / 'vae.pt')
print("VAE complete! Model saved.")